In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


trends = spark.table("scottish_housing_ws.silver.google_trends")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
trends.printSchema()
trends.show(3)

In [0]:
trends_monthly = (
    trends
    .filter(F.col("is_partial") == False)
    .groupBy("year_month")
    .agg(
        F.round(F.avg("house_prices_scotland"), 1).alias("house_prices_scotland"),
        F.round(F.avg("mortgage_edinburgh"), 1).alias("mortgage_edinburgh"),
        F.round(F.avg("rightmove_scotland"), 1).alias("rightmove_scotland"),
        F.round(F.avg("espc_edinburgh"), 1).alias("espc_edinburgh"),
        F.round(F.avg("property_for_sale_edinburgh"), 1).alias("property_for_sale_edinburgh")
    )
)

In [0]:
hpi_filtered = (
    hpi
    .filter(F.col("area_code").isin("S92000003", "S12000036"))
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        "average_price",
        "sales_volume"
    )
)

In [0]:
gold = (
    hpi_filtered
    .join(trends_monthly, on="year_month", how="inner")
    .withColumn(
        "composite_search_index",
        F.round(
            (
                F.col("house_prices_scotland") +
                F.col("rightmove_scotland") +
                F.col("property_for_sale_edinburgh")
            ) / F.lit(3),
            1
        )
    )
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        "average_price",
        "sales_volume",
        "house_prices_scotland",
        "mortgage_edinburgh",
        "rightmove_scotland",
        "espc_edinburgh",
        "property_for_sale_edinburgh",
        "composite_search_index"
    )
    .orderBy("area_code", "report_date")
)

In [0]:
print(f"Row count: {gold.count()}")
gold.filter(F.col("area_code") == "S92000003").orderBy("report_date").show(5)
gold.filter(F.col("area_code") == "S92000003").orderBy(F.col("report_date").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.google_trends_vs_hpi")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_google_trends_vs_hpi/")
)

print("Gold 10 complete.")

In [0]:
# Aggregate weekly trends to monthly 
# Average the index values across weeks within each month
# Exclude partial weeks from the average -- they would drag down
# the current month's value since they represent incomplete data
# is_partial flags the current incomplete week at time of data pull